In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sb
import skimage
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scipy.ndimage as ndimage

import mnds

In [ ]:
# Change dir accordingly:
dir = '/home/caicedo/scr/jcaicedo/Micronuclei-data/'
images_dir = dir + 'dataset_v2/'

files = os.listdir(images_dir)
annot_files = [x for x in files if x.endswith('png')]

In [ ]:
def display_data(im, mni):
    # Show image
    fig, ax = plt.subplots(figsize=(30,30))
    ax.imshow(im)

    # Display micronucleus boxes
    w,h = 16,16
    for k,r in mni.iterrows():
        x1 = r.x - w
        y1 = r.y - h
        rect = patches.Rectangle((x1, y1), 2*w, 2*h, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)

    #plt.axis('off')
    plt.show()


In [ ]:
count_annotations = 0
mni_dfs = []
for fname in annot_files:
    imid = fname.split('.')[0]
    print(imid)
    im = mnds.read_image(images_dir, imid, 'phenotype.tif')
    mni = mnds.read_micronuclei_annotations(images_dir, imid, scale_factor=1.0)
    
    count_annotations += len(mni)
    
    print(f"{imid}: micronuclei:{len(mni)}")
    display_data(im, mni)
    
    mni["Image"] = imid
    mni_dfs.append(mni)
    
print("Total micronuclei:",count_annotations)

In [ ]:
MNI = pd.concat(mni_dfs)

In [ ]:
bins = [x for x in range(0,100,2)]
sb.histplot(data=MNI, x="area", bins=bins)

In [ ]:
bins = [x for x in range(0,800,10)]
sb.histplot(data=MNI, x="area", bins=bins)

In [ ]:
patch = skimage.exposure.rescale_intensity(im, out_range=np.float32)
sobel = skimage.filters.sobel(patch)
sobel = 2*skimage.exposure.rescale_intensity(sobel, out_range=np.float32)
sobel[sobel > 1] = 1
px = np.concatenate(
    (sobel[:,:,np.newaxis], patch[:,:,np.newaxis], patch[:,:,np.newaxis]), 
    axis=2)

In [ ]:
plt.imshow(px[0:256,700:956])